In [8]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [9]:
def generate_random_bits(num_bits):
    """
    Generates a list of random bits (0 or 1) using a quantum measurement.
    The random bits are obtained by preparing a qubit in a superposition state
    and measuring it.

    Args:
        num_bits (int): The number of random bits to generate.

    Returns:
        list: A list of integers (0 or 1) representing the random bits.
    """
    simulator = BasicSimulator()
    random_bits = []
    for _ in range(num_bits):
        # Prepare a qubit in a superposition state |+> = 1/sqrt(2) (|0> + |1>)
        qc = QuantumCircuit(1, 1)
        qc.h(0) # Apply Hadamard gate to create superposition
        qc.measure(0, 0) # Measure in the computational basis

        # Execute the circuit and get results
        compiled_circuit = transpile(qc, simulator)
        job = simulator.run(compiled_circuit, shots=1)
        result = job.result()
        counts = result.get_counts(qc)

        # Extract the measured bit (0 or 1)
        if '0' in counts: # If '0' was measured
            random_bits.append(0)
        else: # If '1' was measured
            random_bits.append(1)
    return random_bits

# Test the random bit generator
# print("Generated 10 random bits:", generate_random_bits(10))

In [10]:
def alice_prepares_qubits(message_length):
    """
    Alice generates random message bits and random bases, then prepares qubits.

    Args:
        message_length (int): The number of bits in Alice's message.

    Returns:
        tuple: A tuple containing:
            - message_bits (list): Alice's random message bits.
            - alice_bases (list): Alice's random choice of bases (0 for computational, 1 for Hadamard).
            - prepared_qubits (list): A list of QuantumCircuit objects, each representing a prepared qubit.
    """
    # Alice generates her random message bits
    message_bits = generate_random_bits(message_length)

    # Alice generates her random choice of bases (0 for Z-basis/computational, 1 for X-basis/Hadamard)
    alice_bases = generate_random_bits(message_length)

    prepared_qubits = []
    for i in range(message_length):
        qc = QuantumCircuit(1, 1) # Create a quantum circuit with 1 qubit and 1 classical bit
        if message_bits[i] == 1: # If the message bit is 1, apply X gate
            qc.x(0)
        if alice_bases[i] == 1: # If the basis is Hadamard, apply H gate
            qc.h(0)
        prepared_qubits.append(qc)

    return message_bits, alice_bases, prepared_qubits

# Example usage (uncomment to test):
# msg_len = 10
# alice_msg_bits, alice_bases_choices, alice_qbs = alice_prepares_qubits(msg_len)
# print("Alice's message bits:", alice_msg_bits)
# print("Alice's bases choices:", alice_bases_choices)
# print("Number of prepared qubits:", len(alice_qbs))

In [11]:
def bob_measures_qubits(prepared_qubits):
    """
    Bob receives the qubits, generates random bases, measures them, and records the results.

    Args:
        prepared_qubits (list): A list of QuantumCircuit objects prepared by Alice.

    Returns:
        tuple: A tuple containing:
            - bob_bases (list): Bob's random choice of bases (0 for computational, 1 for Hadamard).
            - bob_measurement_results (list): Bob's measurement results (0 or 1).
    """
    num_qubits = len(prepared_qubits)
    # Bob generates his random choice of bases
    bob_bases = generate_random_bits(num_qubits)

    bob_measurement_results = []
    simulator = BasicSimulator()

    for i in range(num_qubits):
        # Get the qubit circuit prepared by Alice
        qc = prepared_qubits[i]

        # Apply Bob's chosen basis transformation before measurement
        if bob_bases[i] == 1: # If Bob chose Hadamard basis
            qc.h(0)

        # Measure the qubit
        qc.measure(0, 0)

        # Execute the circuit and get results
        compiled_circuit = transpile(qc, simulator)
        job = simulator.run(compiled_circuit, shots=1)
        result = job.result()
        counts = result.get_counts(qc)

        # Extract the measured bit
        if '0' in counts:
            bob_measurement_results.append(0)
        else:
            bob_measurement_results.append(1)

    return bob_bases, bob_measurement_results

# Example usage (uncomment to test):
# msg_len = 10
# _, _, alice_qbs = alice_prepares_qubits(msg_len)
# bob_bases_choices, bob_results = bob_measures_qubits(alice_qbs)
# print("Bob's bases choices:", bob_bases_choices)
# print("Bob's measurement results:", bob_results)

In [12]:
def reconcile_keys(alice_bases, bob_bases, bob_measurement_results):
    """
    Alice and Bob publicly compare their bases and distill the raw key.

    Args:
        alice_bases (list): Alice's chosen bases.
        bob_bases (list): Bob's chosen bases.
        bob_measurement_results (list): Bob's measurement results.

    Returns:
        list: The raw shared key where Alice's and Bob's bases matched.
    """
    raw_key = []
    matching_indices = []
    for i in range(len(alice_bases)):
        if alice_bases[i] == bob_bases[i]:
            raw_key.append(bob_measurement_results[i])
            matching_indices.append(i)

    return raw_key, matching_indices

# Example usage (uncomment to test):
# msg_len = 20
# alice_msg_bits, alice_bases_choices, alice_qbs = alice_prepares_qubits(msg_len)
# bob_bases_choices, bob_results = bob_measures_qubits(alice_qbs)
# raw_key, matched_indices = reconcile_keys(alice_bases_choices, bob_bases_choices, bob_results)
# print("Alice's original message bits (for matching bases only):", [alice_msg_bits[i] for i in matched_indices])
# print("Bob's measurement results (raw key):", raw_key)
# print("Number of bits in raw key:", len(raw_key))